In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../../../data/processed/realestate_clean_name_ppl.csv')

In [3]:
df

,price,land_area,address_line_2,latitude,longitude,price_per_m2,geometry,index_right,population,h_id,...,Phsar_Chas_1_2km,Phsar_Chas_2_3km,Phsar_Chas_3_5km,Phsar_Chas_5_10km,near_Phsar_kandal_in_km,Phsar_kandal_nearest,Phsar_kandal_1_2km,Phsar_kandal_2_3km,Phsar_kandal_3_5km,Phsar_kandal_5_10km
0,1100000.0,124.0,Chakto Mukh,11.575610,104.920250,8870.967742,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,0,0,0,0,1,0,1,0,0,0
1,1100000.0,124.0,Chakto Mukh,11.575610,104.920250,8870.967742,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,0,0,0,0,1,0,1,0,0,0
2,1100000.0,124.0,Chakto Mukh,11.575610,104.920250,8870.967742,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,0,0,0,0,1,0,1,0,0,0
3,1100000.0,124.0,Chakto Mukh,11.575610,104.920250,8870.967742,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,0,0,0,0,1,0,1,0,0,0
4,1100000.0,124.0,Chakto Mukh,11.575610,104.920250,8870.967742,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,0,0,0,0,1,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7160,550000.0,550000.0,Boeng Reang,11.575837,104.920096,1.000000,POINT (104.920096 11.575837),53920.0,16252.0,8865846aadfffff,...,0,0,0,0,1,0,1,0,0,0
7161,550000.0,550000.0,Boeng Reang,11.575837,104.920096,1.000000,POINT (104.920096 11.575837),53920.0,16252.0,8865846aadfffff,...,0,0,0,0,1,0,1,0,0,0
7162,550000.0,550000.0,Boeng Reang,11.575837,104.920096,1.000000,POINT (104.920096 11.575837),53920.0,16252.0,8865846aadfffff,...,0,0,0,0,1,0,1,0,0,0
7163,550000.0,550000.0,Boeng Reang,11.575837,104.920096,1.000000,POINT (104.920096 11.575837),53920.0,16252.0,8865846aadfffff,...,0,0,0,0,1,0,1,0,0,0


In [4]:
# category_counts_df = df['category_name'].value_counts().reset_index()
# category_counts_df.columns = ['category_name', 'count']
# print(category_counts_df)

In [5]:
# df['price_per_m2'] = df['price'] / df['land_area']

In [6]:
# # Drop rows where category_name is 'Land' or 'House'
# df = df[~df['category_name'].isin(['other'])]

# # Show the updated category counts
# category_counts_df = df['category_name'].value_counts().reset_index()
# category_counts_df.columns = ['category_name', 'count']
# print(category_counts_df)

merge cafe

In [7]:
koi_location = pd.read_csv('../../../data/raw/scrape/koi_the_lat_lon.csv')
brown_cafe_location = pd.read_csv('../../../data/raw/scrape/brown_cafe_lat_lon.csv')
cafe_amazon_location = pd.read_csv('../../../data/raw/scrape/cafe_amazon_lat_lon.csv')
starbuck_location = pd.read_csv('../../../data/raw/scrape/starbuck_lat_lon.csv')
tubecafe_location = pd.read_csv('../../../data/raw/scrape/tubecafe_lat_lon.csv')

In [8]:
starbuck_location.head()

,name,lat,lon
0,ស្ដារប័កស៍ | ប្រៃសណីយ៍,11.575413,104.925674
1,Starbucks,11.565778,104.925339
2,ស្ដារប័កស៍ | Exchange Square,11.573680,104.920977
3,ស្ដារប័កស៍ | មាត់ទន្លេ ៣១៣,11.568047,104.930618
4,Starbucks Reserve BKK Flagship Store,11.553967,104.925008


In [9]:
cafe_df = pd.concat([koi_location, tubecafe_location, cafe_amazon_location, brown_cafe_location, starbuck_location], ignore_index=True)

# Show result
print(cafe_df.tail())

                                               name        lat         lon
291                        ស្ដារប័កស៍ | ស្ទឹងមានជ័យ  11.534940  104.885137
292                           ស្ដារប័កស៍ | សួនអឺរ៉ូ  11.516756  104.956807
293                      ស្ដារប័កស៍ | អ៊ីអន​មាន​ជ័យ  11.483612  104.918262
294  Coffee Concepts (Cambodia) Limited (STARBUCKS)  11.553218  104.940177
295                          ACLEDA ATM - Starbucks  11.588655  104.927458


In [10]:
import numpy as np
cafe_df.drop_duplicates(inplace=True)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

cafe_lats = cafe_df['lat'].values
cafe_lons = cafe_df['lon'].values

def count_cafes(row):
    dists = haversine(row['latitude'], row['longitude'], cafe_lats, cafe_lons)
    return pd.Series({
        'n_cafe_5km': np.sum(dists <= 5),
        'nearest_cafe': np.sum(dists <= 0.5),
        'n_cafe_in_1km': np.sum((dists > 0.5) & (dists <= 1)),
        'n_cafe_in_1km_to_2km': np.sum((dists > 1) & (dists <= 2)),
        'n_cafe_in_2km_to_3km': np.sum((dists > 2) & (dists <= 3)),
        'n_cafe_in_3km_to_5km': np.sum((dists > 3) & (dists <= 5)),
    })

df[['n_cafe_5km',
    'nearest_cafe', 
    'n_cafe_in_1km', 
    'n_cafe_in_1km_to_2km', 
    'n_cafe_in_2km_to_3km', 
    'n_cafe_in_3km_to_5km']] = df.apply(count_cafes, axis=1)
df.drop_duplicates(inplace=True)

In [11]:
df

,price,land_area,address_line_2,latitude,longitude,price_per_m2,geometry,index_right,population,h_id,...,Phsar_kandal_1_2km,Phsar_kandal_2_3km,Phsar_kandal_3_5km,Phsar_kandal_5_10km,n_cafe_5km,nearest_cafe,n_cafe_in_1km,n_cafe_in_1km_to_2km,n_cafe_in_2km_to_3km,n_cafe_in_3km_to_5km
0,1100000.0,124.0,Chakto Mukh,11.575610,104.920250,8870.967742,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,1,0,0,0,174,11,10,33,44,76
6,680000.0,80.0,Boeng Keng Kang Ti Bei,11.550000,104.930000,8500.000000,POINT (104.93 11.55),53894.0,7658.0,8865846ae9fffff,...,0,1,0,0,164,7,14,32,49,62
8,550000.0,66.0,Chey Chumneah,11.575610,104.920250,8333.333333,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,1,0,0,0,174,11,10,33,44,76
14,750000.0,116.0,Tonle Basak,11.544500,104.913586,6465.517241,POINT (104.913586 11.5445),53909.0,23239.0,8865846ac7fffff,...,0,0,1,0,177,7,12,36,37,85
20,420000.0,65.0,Chrouy Changvar,11.580000,104.930000,6461.538462,POINT (104.93 11.58),54133.0,5351.0,886584685bfffff,...,1,0,0,0,157,0,6,27,37,87
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7147,18000.0,1400.0,Phnom Penh Thmei,11.575610,104.920250,12.857143,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,1,0,0,0,174,11,10,33,44,76
7153,3400000.0,400000.0,Preaek Aeng,11.522551,104.962474,8.500000,POINT (104.962474 11.522551),53953.0,3342.0,8865846a51fffff,...,0,0,0,1,49,0,1,8,4,36
7155,35000.0,10000.0,Chrouy Changvar,11.589674,104.925654,3.500000,POINT (104.925654 11.589674),54162.0,11327.0,8865846819fffff,...,0,1,0,0,141,4,3,19,29,86
7158,270000.0,270000.0,Kamboul,11.531222,104.776086,1.000000,POINT (104.776086 11.5312221),53668.0,1122.0,8865846e33fffff,...,0,0,0,0,3,0,0,0,0,3


In [12]:
# cafe_df.to_csv('../../data/raw/cafe_location.csv', index=False)

merge gas station

In [13]:
total_location = pd.read_csv('../../../data/raw/scrape/total_lat_lon.csv')
ptt_location = pd.read_csv('../../../data/raw/scrape/ptt_lat_lon.csv')
caltex_location = pd.read_csv('../../../data/raw/scrape/caltex_lat_lon.csv')


In [14]:
gas_station_df = pd.concat([total_location, ptt_location, caltex_location], ignore_index=True)


In [15]:
gas_station_df.drop_duplicates(inplace=True)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

gas_station_lats = gas_station_df['lat'].values
gas_station_lons = gas_station_df['lon'].values

def count_gas_station(row):
    dists = haversine(row['latitude'], row['longitude'], gas_station_lats, gas_station_lons)
    return pd.Series({
        'n_gas_station_5km': np.sum(dists <= 5),
        'nearest_gas_station': np.sum(dists <= 0.5),
        'n_gas_station_in_1km': np.sum((dists > 0.5) & (dists <= 1)),
        'n_gas_station_in_1km_to_2km': np.sum((dists > 1) & (dists <= 2)),
        'n_gas_station_in_2km_to_3km': np.sum((dists > 2) & (dists <= 3)),
        'n_gas_station_in_3km_to_5km': np.sum((dists > 3) & (dists <= 5)),
    })

df[['n_gas_station_5km',
    'nearest_gas_station', 
    'n_gas_station_in_1km', 
    'n_gas_station_in_1km_to_2km', 
    'n_gas_station_in_2km_to_3km', 
    'n_gas_station_in_3km_to_5km']] = df.apply(count_gas_station, axis=1)

df.drop_duplicates(inplace=True)

In [16]:
# gas_station_df.to_csv('../../data/raw/gas_station_location.csv', index=False)

Near Hospital

In [17]:
hospital_df = pd.read_csv('../../../data/raw/scrape/hospital_lat_lon.csv')
hospital_df.drop_duplicates(inplace=True)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

hospital_lats = hospital_df['lat'].values
hospital_lons = hospital_df['lon'].values

def count_hospital(row):
    dists = haversine(row['latitude'], row['longitude'], hospital_lats, hospital_lons)
    return pd.Series({
        'n_hospital_5km': np.sum(dists <= 5),
        'nearest_hospital': np.sum(dists <= 0.5),
        'n_hospital_in_1km': np.sum((dists > 0.5) & (dists <= 1)),
        'n_hospital_in_1km_to_2km': np.sum((dists > 1) & (dists <= 2)),
        'n_hospital_in_2km_to_3km': np.sum((dists > 2) & (dists <= 3)),
        'n_hospital_in_3km_to_5km': np.sum((dists > 3) & (dists <= 5)),
    })

df[['n_hospital_5km',
    'nearest_hospital', 
    'n_hospital_in_1km', 
    'n_hospital_in_1km_to_2km', 
    'n_hospital_in_2km_to_3km', 
    'n_hospital_in_3km_to_5km']] = df.apply(count_hospital, axis=1)

df.drop_duplicates(inplace=True)

near hotel

In [18]:
hotel_df = pd.read_csv('../../../data/raw/scrape/hotel_lat_lon.csv')
hotel_df.drop_duplicates(inplace=True)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

hotel_lats = hotel_df['lat'].values
hotel_lons = hotel_df['lon'].values

def count_hotel(row):
    dists = haversine(row['latitude'], row['longitude'], hotel_lats, hotel_lons)
    return pd.Series({
        'n_hotel_5km': np.sum(dists <= 5),
        'nearest_hotel': np.sum(dists <= 0.5),
        'n_hotel_in_1km': np.sum((dists > 0.5) & (dists <= 1)),
        'n_hotel_in_1km_to_2km': np.sum((dists > 1) & (dists <= 2)),
        'n_hotel_in_2km_to_3km': np.sum((dists > 2) & (dists <= 3)),
        'n_hotel_in_3km_to_5km': np.sum((dists > 3) & (dists <= 5)),
    })

df[['n_hotel_5km',
    'nearest_hotel', 
    'n_hotel_in_1km', 
    'n_hotel_in_1km_to_2km', 
    'n_hotel_in_2km_to_3km', 
    'n_hotel_in_3km_to_5km']] = df.apply(count_hotel, axis=1)

df.drop_duplicates(inplace=True)


near mart

In [19]:
mart_df = pd.read_csv('../../../data/raw/scrape/mart_lat_lon.csv')
mart_df.drop_duplicates(inplace=True)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

mart_lats = mart_df['lat'].values
mart_lons = mart_df['lon'].values

def count_mart(row):
    dists = haversine(row['latitude'], row['longitude'], mart_lats, mart_lons)
    return pd.Series({
        'n_mart_5km': np.sum(dists <= 5),
        'nearest_mart': np.sum(dists <= 0.5),
        'n_mart_in_1km': np.sum((dists > 0.5) & (dists <= 1)),
        'n_mart_in_1km_to_2km': np.sum((dists > 1) & (dists <= 2)),
        'n_mart_in_2km_to_3km': np.sum((dists > 2) & (dists <= 3)),
        'n_mart_in_3km_to_5km': np.sum((dists > 3) & (dists <= 5)),
    })

df[['n_mart_5km',
    'nearest_mart', 
    'n_mart_in_1km', 
    'n_mart_in_1km_to_2km', 
    'n_mart_in_2km_to_3km', 
    'n_mart_in_3km_to_5km']] = df.apply(count_mart, axis=1)

df.drop_duplicates(inplace=True)


near pre school

In [20]:
pre_school_df = pd.read_csv('../../../data/raw/scrape/pre_school_lat_lon.csv')
pre_school_df.drop_duplicates(inplace=True)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

pre_school_lats = pre_school_df['lat'].values
pre_school_lons = pre_school_df['lon'].values

def count_pre_school(row):
    dists = haversine(row['latitude'], row['longitude'], pre_school_lats, pre_school_lons)
    return pd.Series({
        'n_pre_school_5km': np.sum(dists <= 5),
        'nearest_pre_school': np.sum(dists <= 0.5),
        'n_pre_school_in_1km': np.sum((dists > 0.5) & (dists <= 1)),
        'n_pre_school_in_1km_to_2km': np.sum((dists > 1) & (dists <= 2)),
        'n_pre_school_in_2km_to_3km': np.sum((dists > 2) & (dists <= 3)),
        'n_pre_school_in_3km_to_5km': np.sum((dists > 3) & (dists <= 5)),
    })

df[['n_pre_school_5km',
    'nearest_pre_school', 
    'n_pre_school_in_1km', 
    'n_pre_school_in_1km_to_2km', 
    'n_pre_school_in_2km_to_3km', 
    'n_pre_school_in_3km_to_5km']] = df.apply(count_pre_school, axis=1)

df.drop_duplicates(inplace=True)

secondary school

In [21]:
secondary_school_df = pd.read_csv('../../../data/raw/scrape/secondary_school_lat_lon.csv')
secondary_school_df.drop_duplicates(inplace=True)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

secondary_school_lats = secondary_school_df['lat'].values
secondary_school_lons = secondary_school_df['lon'].values

def count_secondary_school(row):
    dists = haversine(row['latitude'], row['longitude'], secondary_school_lats, secondary_school_lons)
    return pd.Series({
        'n_secondary_school_5km': np.sum(dists <= 5),
        'nearest_secondary_school': np.sum(dists <= 0.5),
        'n_secondary_school_in_1km': np.sum((dists > 0.5) & (dists <= 1)),
        'n_secondary_school_in_1km_to_2km': np.sum((dists > 1) & (dists <= 2)),
        'n_secondary_school_in_2km_to_3km': np.sum((dists > 2) & (dists <= 3)),
        'n_secondary_school_in_3km_to_5km': np.sum((dists > 3) & (dists <= 5)),
    })

df[['n_secondary_school_5km',
    'nearest_secondary_school', 
    'n_secondary_school_in_1km', 
    'n_secondary_school_in_1km_to_2km', 
    'n_secondary_school_in_2km_to_3km', 
    'n_secondary_school_in_3km_to_5km']] = df.apply(count_secondary_school, axis=1)

df.drop_duplicates(inplace=True)


primary school

In [22]:
primary_school_df = pd.read_csv('../../../data/raw/scrape/primary_school_lat_lon.csv')
primary_school_df.drop_duplicates(inplace=True)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

primary_school_lats = primary_school_df['lat'].values
primary_school_lons = primary_school_df['lon'].values

def count_primary_school(row):
    dists = haversine(row['latitude'], row['longitude'], primary_school_lats, primary_school_lons)
    return pd.Series({
        'n_primary_school_5km': np.sum(dists <= 5),
        'nearest_primary_school': np.sum(dists <= 0.5),
        'n_primary_school_in_1km': np.sum((dists > 0.5) & (dists <= 1)),
        'n_primary_school_in_1km_to_2km': np.sum((dists > 1) & (dists <= 2)),
        'n_primary_school_in_2km_to_3km': np.sum((dists > 2) & (dists <= 3)),
        'n_primary_school_in_3km_to_5km': np.sum((dists > 3) & (dists <= 5)),
    })

df[['n_primary_school_5km',
    'nearest_primary_school', 
    'n_primary_school_in_1km', 
    'n_primary_school_in_1km_to_2km', 
    'n_primary_school_in_2km_to_3km', 
    'n_primary_school_in_3km_to_5km']] = df.apply(count_primary_school, axis=1)

df.drop_duplicates(inplace=True)


In [23]:
df

,price,land_area,address_line_2,latitude,longitude,price_per_m2,geometry,index_right,population,h_id,...,n_secondary_school_in_1km,n_secondary_school_in_1km_to_2km,n_secondary_school_in_2km_to_3km,n_secondary_school_in_3km_to_5km,n_primary_school_5km,nearest_primary_school,n_primary_school_in_1km,n_primary_school_in_1km_to_2km,n_primary_school_in_2km_to_3km,n_primary_school_in_3km_to_5km
0,1100000.0,124.0,Chakto Mukh,11.575610,104.920250,8870.967742,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,2,22,20,32,64,1,2,13,20,28
6,680000.0,80.0,Boeng Keng Kang Ti Bei,11.550000,104.930000,8500.000000,POINT (104.93 11.55),53894.0,7658.0,8865846ae9fffff,...,3,16,16,34,63,0,4,13,15,31
8,550000.0,66.0,Chey Chumneah,11.575610,104.920250,8333.333333,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,2,22,20,32,64,1,2,13,20,28
14,750000.0,116.0,Tonle Basak,11.544500,104.913586,6465.517241,POINT (104.913586 11.5445),53909.0,23239.0,8865846ac7fffff,...,4,14,30,22,79,2,4,17,21,35
20,420000.0,65.0,Chrouy Changvar,11.580000,104.930000,6461.538462,POINT (104.93 11.58),54133.0,5351.0,886584685bfffff,...,2,14,17,32,52,0,0,9,10,33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7147,18000.0,1400.0,Phnom Penh Thmei,11.575610,104.920250,12.857143,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,2,22,20,32,64,1,2,13,20,28
7153,3400000.0,400000.0,Preaek Aeng,11.522551,104.962474,8.500000,POINT (104.962474 11.522551),53953.0,3342.0,8865846a51fffff,...,0,1,4,8,10,0,0,0,3,7
7155,35000.0,10000.0,Chrouy Changvar,11.589674,104.925654,3.500000,POINT (104.925654 11.589674),54162.0,11327.0,8865846819fffff,...,3,9,13,35,49,0,2,4,6,37
7158,270000.0,270000.0,Kamboul,11.531222,104.776086,1.000000,POINT (104.776086 11.5312221),53668.0,1122.0,8865846e33fffff,...,1,0,0,2,1,0,0,0,0,1


In [24]:
df.drop_duplicates(inplace=True)

near university

In [25]:
university_df = pd.read_csv('../../../data/raw/scrape/university_lat_lon.csv')
university_df.drop_duplicates(inplace=True)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

university_lats = university_df['lat'].values
university_lons = university_df['lon'].values

def count_university(row):
    dists = haversine(row['latitude'], row['longitude'], university_lats, university_lons)
    return pd.Series({
        'n_university_5km': np.sum(dists <= 5),
        'nearest_university': np.sum(dists <= 0.5),
        'n_university_in_1km': np.sum((dists > 0.5) & (dists <= 1)),
        'n_university_in_1km_to_2km': np.sum((dists > 1) & (dists <= 2)),
        'n_university_in_2km_to_3km': np.sum((dists > 2) & (dists <= 3)),
        'n_university_in_3km_to_5km': np.sum((dists > 3) & (dists <= 5)),
    })

df[['n_university_5km',
    'nearest_university', 
    'n_university_in_1km', 
    'n_university_in_1km_to_2km', 
    'n_university_in_2km_to_3km', 
    'n_university_in_3km_to_5km']] = df.apply(count_university, axis=1)

df.drop_duplicates(inplace=True)


near seven_eleven

In [26]:
seven_eleven_df = pd.read_csv('../../../data/raw/scrape/sevenevelen_lat_lon.csv')
seven_eleven_df.drop_duplicates(inplace=True)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

seven_eleven_lats = seven_eleven_df['lat'].values
seven_eleven_lons = seven_eleven_df['lon'].values

def count_seven_eleven(row):
    dists = haversine(row['latitude'], row['longitude'], seven_eleven_lats, seven_eleven_lons)
    return pd.Series({
        'n_seven_eleven_5km': np.sum(dists <= 5),
        'nearest_seven_eleven': np.sum(dists <= 0.5),
        'n_seven_eleven_in_1km': np.sum((dists > 0.5) & (dists <= 1)),
        'n_seven_eleven_in_1km_to_2km': np.sum((dists > 1) & (dists <= 2)),
        'n_seven_eleven_in_2km_to_3km': np.sum((dists > 2) & (dists <= 3)),
        'n_seven_eleven_in_3km_to_5km': np.sum((dists > 3) & (dists <= 5)),
    })

df[['n_seven_eleven_5km',
    'nearest_seven_eleven', 
    'n_seven_eleven_in_1km', 
    'n_seven_eleven_in_1km_to_2km', 
    'n_seven_eleven_in_2km_to_3km', 
    'n_seven_eleven_in_3km_to_5km']] = df.apply(count_seven_eleven, axis=1)

df.drop_duplicates(inplace=True)


Near Resturant

In [27]:
resturant_df = pd.read_csv('../../../data/raw/scrape/resturant_lat_lon.csv')
resturant_df.drop_duplicates(inplace=True)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

resturant_lats = resturant_df['lat'].values
resturant_lons = resturant_df['lon'].values

def count_resturant(row):
    dists = haversine(row['latitude'], row['longitude'], resturant_lats, resturant_lons)
    return pd.Series({
        'n_resturant_5km': np.sum(dists <= 5),
        'nearest_resturant': np.sum(dists <= 0.5),
        'n_resturant_in_1km': np.sum((dists > 0.5) & (dists <= 1)),
        'n_resturant_in_1km_to_2km': np.sum((dists > 1) & (dists <= 2)),
        'n_resturant_in_2km_to_3km': np.sum((dists > 2) & (dists <= 3)),
        'n_resturant_in_3km_to_5km': np.sum((dists > 3) & (dists <= 5)),
    })

df[['n_resturant_5km',
    'nearest_resturant', 
    'n_resturant_in_1km', 
    'n_resturant_in_1km_to_2km', 
    'n_resturant_in_2km_to_3km', 
    'n_resturant_in_3km_to_5km']] = df.apply(count_resturant, axis=1)

df.drop_duplicates(inplace=True)


super market

In [28]:
super_market_df = pd.read_csv('../../../data/raw/scrape/super_market_lat_lon.csv')
super_market_df.drop_duplicates(inplace=True)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

super_market_lats = super_market_df['lat'].values
super_market_lons = super_market_df['lon'].values

def count_super_market(row):
    dists = haversine(row['latitude'], row['longitude'], super_market_lats, super_market_lons)
    return pd.Series({
        'n_super_market_5km': np.sum(dists <= 5),
        'nearest_super_market': np.sum(dists <= 0.5),
        'n_super_market_in_1km': np.sum((dists > 0.5) & (dists <= 1)),
        'n_super_market_in_1km_to_2km': np.sum((dists > 1) & (dists <= 2)),
        'n_super_market_in_2km_to_3km': np.sum((dists > 2) & (dists <= 3)),
        'n_super_market_in_3km_to_5km': np.sum((dists > 3) & (dists <= 5)),
    })

df[['n_super_market_5km',
    'nearest_super_market', 
    'n_super_market_in_1km', 
    'n_super_market_in_1km_to_2km', 
    'n_super_market_in_2km_to_3km', 
    'n_super_market_in_3km_to_5km']] = df.apply(count_super_market, axis=1)

df.drop_duplicates(inplace=True)


In [29]:
df

,price,land_area,address_line_2,latitude,longitude,price_per_m2,geometry,index_right,population,h_id,...,n_resturant_in_1km,n_resturant_in_1km_to_2km,n_resturant_in_2km_to_3km,n_resturant_in_3km_to_5km,n_super_market_5km,nearest_super_market,n_super_market_in_1km,n_super_market_in_1km_to_2km,n_super_market_in_2km_to_3km,n_super_market_in_3km_to_5km
0,1100000.0,124.0,Chakto Mukh,11.575610,104.920250,8870.967742,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,47,42,19,8,97,1,14,28,25,29
6,680000.0,80.0,Boeng Keng Kang Ti Bei,11.550000,104.930000,8500.000000,POINT (104.93 11.55),53894.0,7658.0,8865846ae9fffff,...,7,34,64,4,88,2,13,23,30,20
8,550000.0,66.0,Chey Chumneah,11.575610,104.920250,8333.333333,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,47,42,19,8,97,1,14,28,25,29
14,750000.0,116.0,Tonle Basak,11.544500,104.913586,6465.517241,POINT (104.913586 11.5445),53909.0,23239.0,8865846ac7fffff,...,4,18,28,68,95,2,9,20,29,35
20,420000.0,65.0,Chrouy Changvar,11.580000,104.930000,6461.538462,POINT (104.93 11.58),54133.0,5351.0,886584685bfffff,...,35,49,18,16,87,0,4,23,26,34
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7147,18000.0,1400.0,Phnom Penh Thmei,11.575610,104.920250,12.857143,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,47,42,19,8,97,1,14,28,25,29
7153,3400000.0,400000.0,Preaek Aeng,11.522551,104.962474,8.500000,POINT (104.962474 11.522551),53953.0,3342.0,8865846a51fffff,...,0,0,0,9,11,0,0,2,0,9
7155,35000.0,10000.0,Chrouy Changvar,11.589674,104.925654,3.500000,POINT (104.925654 11.589674),54162.0,11327.0,8865846819fffff,...,0,32,52,29,81,2,2,3,27,47
7158,270000.0,270000.0,Kamboul,11.531222,104.776086,1.000000,POINT (104.776086 11.5312221),53668.0,1122.0,8865846e33fffff,...,0,0,0,0,0,0,0,0,0,0


near borey

In [30]:
borey_df = pd.read_csv('../../../data/raw/scrape/borey_lat_lon.csv')
borey_df.drop_duplicates(inplace=True)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

borey_lats = borey_df['lat'].values
borey_lons = borey_df['lon'].values

def count_borey(row):
    dists = haversine(row['latitude'], row['longitude'], borey_lats, borey_lons)
    return pd.Series({
        'n_borey_5km': np.sum(dists <= 5),
        'nearest_borey': np.sum(dists <= 0.5),
        'n_borey_in_1km': np.sum((dists > 0.5) & (dists <= 1)),
        'n_borey_in_1km_to_2km': np.sum((dists > 1) & (dists <= 2)),
        'n_borey_in_2km_to_3km': np.sum((dists > 2) & (dists <= 3)),
        'n_borey_in_3km_to_5km': np.sum((dists > 3) & (dists <= 5)),
    })

df[['n_borey_5km',
    'nearest_borey', 
    'n_borey_in_1km', 
    'n_borey_in_1km_to_2km', 
    'n_borey_in_2km_to_3km', 
    'n_borey_in_3km_to_5km']] = df.apply(count_borey, axis=1)

df.drop_duplicates(inplace=True)


Near bank and ATM

In [31]:
wing_bank_lat_lon = pd.read_csv('../../../data/raw/scrape/wing_bank_lat_lon.csv')
atm_lat_lon = pd.read_csv('../../../data/raw/scrape/atm_lat_lon.csv')
bank_lat_lon = pd.read_csv('../../../data/raw/scrape/bank_lat_lon.csv')
aceleda_bank_lat_lon = pd.read_csv('../../../data/raw/scrape/aceleda_bank_lat_lon.csv')
aba_bank_lat_lon = pd.read_csv('../../../data/raw/scrape/aba_bank_lat_lon.csv')

In [32]:
bank_and_atm_df = pd.concat([wing_bank_lat_lon, atm_lat_lon, bank_lat_lon, aceleda_bank_lat_lon, aba_bank_lat_lon], ignore_index=True)

# Show result
print(bank_and_atm_df)


                                        name        lat         lon  \
0                                 ធនាគារ វីង  11.545046  104.922043   
1             Wing Bank Beoung Trobek Branch  11.541721  104.922546   
2     Wing Bank Independence Monument Branch  11.555800  104.924057   
3                 Wing Bank Wat Phnom Branch  11.571245  104.920211   
4           Wing Bank Preah Yukunthor Branch  11.556008  104.919105   
...                                      ...        ...         ...   
1334               ABA Check Deposit Machine  11.562373  104.912932   
1335                                 ABA ATM  11.549432  104.924192   
1336                                ABA Bank  11.530632  104.857228   
1337                                 ABA ATM  11.548479  104.928196   
1338                     ABA 24/7 - WB Arena  11.510290  104.937350   

      Unnamed: 0.4  Unnamed: 0.3  Unnamed: 0.2  Unnamed: 0.1  Unnamed: 0  
0              NaN           NaN           NaN           NaN         NaN

In [33]:
# Split bank_and_atm_df into two DataFrames

# Bank branches: rows where 'name' contains 'branch' (case-insensitive)
bank_lat_lon = bank_and_atm_df[bank_and_atm_df['name'].str.lower().str.contains('branch')].reset_index(drop=True)

# ATMs: rows where 'name' contains 'ATM' or '24/7' (case-insensitive)
atm_lat_lon = bank_and_atm_df[
    bank_and_atm_df['name'].str.upper().str.contains('ATM') | 
    bank_and_atm_df['name'].str.contains('24/7', case=False)
].reset_index(drop=True)

print("Bank branches:")
print(bank_lat_lon.head())
print("ATMs:")
print(atm_lat_lon.head())

Bank branches:
                                     name        lat         lon  \
0          Wing Bank Beoung Trobek Branch  11.541721  104.922546   
1  Wing Bank Independence Monument Branch  11.555800  104.924057   
2              Wing Bank Wat Phnom Branch  11.571245  104.920211   
3        Wing Bank Preah Yukunthor Branch  11.556008  104.919105   
4              Wing Bank Tuol Kouk Branch  11.578434  104.899991   

   Unnamed: 0.4  Unnamed: 0.3  Unnamed: 0.2  Unnamed: 0.1  Unnamed: 0  
0           NaN           NaN           NaN           NaN         NaN  
1           NaN           NaN           NaN           NaN         NaN  
2           NaN           NaN           NaN           NaN         NaN  
3           NaN           NaN           NaN           NaN         NaN  
4           NaN           NaN           NaN           NaN         NaN  
ATMs:
                       name        lat         lon  Unnamed: 0.4  \
0  ABA 24/7 - Phsar Thmey 2  11.569136  104.918729           NaN   
1 

In [34]:
bank_lat_lon.to_csv('../../../data/raw/scrape/bank_lat_lon.csv')
atm_lat_lon.to_csv('../../../data/raw/scrape/atm_lat_lon.csv')

In [35]:
bank_lat_lon.drop_duplicates(inplace=True)

In [36]:
atm_lat_lon.drop_duplicates(inplace=True)

near bank

In [37]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

bank_lats = bank_lat_lon['lat'].values
bank_lons = bank_lat_lon['lon'].values

def count_bank(row):
    dists = haversine(row['latitude'], row['longitude'], bank_lats, bank_lons)
    return pd.Series({
        'n_bank_5km': np.sum(dists <= 5),
        'nearest_bank': np.sum(dists <= 0.5),
        'n_bank_in_1km': np.sum((dists > 0.5) & (dists <= 1)),
        'n_bank_in_1km_to_2km': np.sum((dists > 1) & (dists <= 2)),
        'n_bank_in_2km_to_3km': np.sum((dists > 2) & (dists <= 3)),
        'n_bank_in_3km_to_5km': np.sum((dists > 3) & (dists <= 5)),
    })

df[['n_bank_5km',
    'nearest_bank', 
    'n_bank_in_1km', 
    'n_bank_in_1km_to_2km', 
    'n_bank_in_2km_to_3km', 
    'n_bank_in_3km_to_5km']] = df.apply(count_bank, axis=1)

df.drop_duplicates(inplace=True)


near atm

In [38]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

atm_lats = atm_lat_lon['lat'].values
atm_lons = atm_lat_lon['lon'].values

def count_atm(row):
    dists = haversine(row['latitude'], row['longitude'], atm_lats, atm_lons)
    return pd.Series({
        'n_atm_5km': np.sum(dists <= 5),
        'nearest_atm': np.sum(dists <= 0.5),
        'n_atm_in_1km': np.sum((dists > 0.5) & (dists <= 1)),
        'n_atm_in_1km_to_2km': np.sum((dists > 1) & (dists <= 2)),
        'n_atm_in_2km_to_3km': np.sum((dists > 2) & (dists <= 3)),
        'n_atm_in_3km_to_5km': np.sum((dists > 3) & (dists <= 5)),
    })

df[['n_atm_5km',
    'nearest_atm', 
    'n_atm_in_1km', 
    'n_atm_in_1km_to_2km', 
    'n_atm_in_2km_to_3km', 
    'n_atm_in_3km_to_5km']] = df.apply(count_atm, axis=1)

df.drop_duplicates(inplace=True)


In [39]:
df.shape

(2399, 214)

In [40]:
df

,price,land_area,address_line_2,latitude,longitude,price_per_m2,geometry,index_right,population,h_id,...,n_bank_in_1km,n_bank_in_1km_to_2km,n_bank_in_2km_to_3km,n_bank_in_3km_to_5km,n_atm_5km,nearest_atm,n_atm_in_1km,n_atm_in_1km_to_2km,n_atm_in_2km_to_3km,n_atm_in_3km_to_5km
0,1100000.0,124.0,Chakto Mukh,11.575610,104.920250,8870.967742,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,79,112,40,30,1027,47,218,414,162,186
6,680000.0,80.0,Boeng Keng Kang Ti Bei,11.550000,104.930000,8500.000000,POINT (104.93 11.55),53894.0,7658.0,8865846ae9fffff,...,7,64,166,14,973,12,84,206,496,175
8,550000.0,66.0,Chey Chumneah,11.575610,104.920250,8333.333333,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,79,112,40,30,1027,47,218,414,162,186
14,750000.0,116.0,Tonle Basak,11.544500,104.913586,6465.517241,POINT (104.913586 11.5445),53909.0,23239.0,8865846ac7fffff,...,0,73,125,60,985,18,36,184,359,388
20,420000.0,65.0,Chrouy Changvar,11.580000,104.930000,6461.538462,POINT (104.93 11.58),54133.0,5351.0,886584685bfffff,...,32,62,127,31,973,12,115,313,317,216
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7147,18000.0,1400.0,Phnom Penh Thmei,11.575610,104.920250,12.857143,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,79,112,40,30,1027,47,218,414,162,186
7153,3400000.0,400000.0,Preaek Aeng,11.522551,104.962474,8.500000,POINT (104.962474 11.522551),53953.0,3342.0,8865846a51fffff,...,0,0,0,19,66,0,0,0,0,66
7155,35000.0,10000.0,Chrouy Changvar,11.589674,104.925654,3.500000,POINT (104.925654 11.589674),54162.0,11327.0,8865846819fffff,...,6,1,101,144,931,12,18,134,362,405
7158,270000.0,270000.0,Kamboul,11.531222,104.776086,1.000000,POINT (104.776086 11.5312221),53668.0,1122.0,8865846e33fffff,...,0,6,0,0,0,0,0,0,0,0


In [41]:
# df = pd.read_csv('../../data/processed/realestates_kh_v6.csv')

In [42]:
aeon_lat_lon = pd.read_csv('../../../data/raw/scrape/aeon_lat_lon.csv')

In [43]:
aeon_lat_lon

,name,lat,lon
0,ផ្សារអ៉ីអន ភ្នំពេញ,11.548192,104.932863
1,​ផ្សារ​អ៊ីអន សែន​សុខ.,11.599664,104.885370
2,AEON MALL Sen Sok,11.598883,104.884577
3,Levi’s (Aeon Mall Phnom Penh),11.548006,104.932861
4,ផ្សារទំនើបសូរិយា,11.567605,104.920839
...,...,...,...
68,Yolé AEON Mall Sensok City,11.599942,104.885507
69,SIMPL'SELF AEON MALL Phnom Penh 1st Floor,11.548082,104.933272
70,TWG Tea at Aeon Mall Phnom Penh,11.548229,104.932796
71,Estelle,11.547569,104.933073


In [44]:
df

,price,land_area,address_line_2,latitude,longitude,price_per_m2,geometry,index_right,population,h_id,...,n_bank_in_1km,n_bank_in_1km_to_2km,n_bank_in_2km_to_3km,n_bank_in_3km_to_5km,n_atm_5km,nearest_atm,n_atm_in_1km,n_atm_in_1km_to_2km,n_atm_in_2km_to_3km,n_atm_in_3km_to_5km
0,1100000.0,124.0,Chakto Mukh,11.575610,104.920250,8870.967742,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,79,112,40,30,1027,47,218,414,162,186
6,680000.0,80.0,Boeng Keng Kang Ti Bei,11.550000,104.930000,8500.000000,POINT (104.93 11.55),53894.0,7658.0,8865846ae9fffff,...,7,64,166,14,973,12,84,206,496,175
8,550000.0,66.0,Chey Chumneah,11.575610,104.920250,8333.333333,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,79,112,40,30,1027,47,218,414,162,186
14,750000.0,116.0,Tonle Basak,11.544500,104.913586,6465.517241,POINT (104.913586 11.5445),53909.0,23239.0,8865846ac7fffff,...,0,73,125,60,985,18,36,184,359,388
20,420000.0,65.0,Chrouy Changvar,11.580000,104.930000,6461.538462,POINT (104.93 11.58),54133.0,5351.0,886584685bfffff,...,32,62,127,31,973,12,115,313,317,216
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7147,18000.0,1400.0,Phnom Penh Thmei,11.575610,104.920250,12.857143,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,...,79,112,40,30,1027,47,218,414,162,186
7153,3400000.0,400000.0,Preaek Aeng,11.522551,104.962474,8.500000,POINT (104.962474 11.522551),53953.0,3342.0,8865846a51fffff,...,0,0,0,19,66,0,0,0,0,66
7155,35000.0,10000.0,Chrouy Changvar,11.589674,104.925654,3.500000,POINT (104.925654 11.589674),54162.0,11327.0,8865846819fffff,...,6,1,101,144,931,12,18,134,362,405
7158,270000.0,270000.0,Kamboul,11.531222,104.776086,1.000000,POINT (104.776086 11.5312221),53668.0,1122.0,8865846e33fffff,...,0,6,0,0,0,0,0,0,0,0


In [45]:
df.to_csv('../../../data/processed/realestate_clean_name_ppl.csv', index=False)

In [46]:
# df2 = pd.read_csv('../../data/processed/realestates_kh_v6.csv')


In [47]:
# df_combined = pd.concat([df, df2], ignore_index=True)

In [48]:
# df_combined.to_csv('../../data/processed/realestates_kh_v7.csv', index=False)

In [49]:
# df.to_csv('../../data/processed/realestates_kh_v5.csv', index=False)